In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score
import xgboost as xgb

RANDOM_STATE = 42
TARGET = "loan_paid_back"

# 类别型特征（XGBoost原生处理，不需要One-Hot）
CAT_COLS = ["gender", "marital_status", "education_level",
            "employment_status", "loan_purpose", "grade_subgrade"]

# 组长指定的4个公平性敏感属性
PROTECTED_ATTRS = ["gender", "marital_status", "employment_status", "education_level"]

In [2]:
# test_eval_20k 是三个模型共用的评估集，train.csv 必须先剔除这2万行，
# 否则你的模型相当于"见过"评估集里的数据，分数会虚高
test_eval = pd.read_csv("test_eval_20k.csv")
train = pd.read_csv("train.csv")                       # Kaggle's train.csv
train = train[~train["id"].isin(test_eval["id"])]       # 剔除共享评估集对应的行

print("训练集(剔除后):", train.shape)   # 应该是 ~573,994 行
print("共享评估集:", test_eval.shape)   # 20,000 行

训练集(剔除后): (573994, 13)
共享评估集: (20000, 13)


In [4]:
# XGBoost需要显式把类别列转成'category'dtype才能用enable_categorical原生处理
for c in CAT_COLS:
    train[c] = train[c].astype("category")
    test_eval[c] = test_eval[c].astype("category")

X_all = train.drop(columns=["id", TARGET])
y_all = train[TARGET].astype(int)

In [5]:
X_fit, X_rest, y_fit, y_rest = train_test_split(
    X_all, y_all, test_size=0.3, stratify=y_all, random_state=RANDOM_STATE
)
X_es, X_thr, y_es, y_thr = train_test_split(
    X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=RANDOM_STATE
)
print(X_fit.shape, X_es.shape, X_thr.shape)

(401795, 11) (86099, 11) (86100, 11)


In [6]:
scale_pos_weight = (y_fit == 0).sum() / (y_fit == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    enable_categorical=True,
    tree_method="hist",
    eval_metric="auc",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
)
model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], verbose=False)
print("early stopping在第", model.best_iteration, "轮停止")

early stopping在第 837 轮停止


In [7]:
def pnl(y_true, y_pred_label, loan_amount, interest_rate):
    """
    组长规定的经济计算规则：
    - 批准且还款: + loan_amount * interest_rate / 100
    - 批准且违约: - loan_amount * 0.60   (假设银行损失60%本金)
    - 拒绝: 0
    """
    approved = (y_pred_label == 1)
    repaid = (y_true == 1)
    defaulted = (y_true == 0)

    profit = (approved & repaid) * (loan_amount * interest_rate / 100)
    loss = (approved & defaulted) * (loan_amount * 0.60)
    return (profit - loss).sum()

In [8]:
thr_scores = model.predict_proba(X_thr)[:, 1]

thresholds = np.arange(0.01, 1.0, 0.01)
pnls = [
    pnl(y_thr.values, (thr_scores >= t).astype(int),
        X_thr["loan_amount"].values, X_thr["interest_rate"].values)
    for t in thresholds
]

best_threshold = thresholds[np.argmax(pnls)]
print(f"最优approval阈值: {best_threshold:.2f}, 对应validation P&L: {max(pnls):,.0f}")

最优approval阈值: 0.48, 对应validation P&L: 76,825,500


In [9]:
X_test = test_eval.drop(columns=["id", TARGET])
y_test = test_eval[TARGET].astype(int)

test_scores = model.predict_proba(X_test)[:, 1]
test_labels = (test_scores >= best_threshold).astype(int)   # 用Cell 7选出的阈值，不重新调参

roc_auc = roc_auc_score(y_test, test_scores)
gini = 2 * roc_auc - 1
pr_auc = average_precision_score(y_test, test_scores)
fpr, tpr, _ = roc_curve(y_test, test_scores)
ks = np.max(np.abs(tpr - fpr))
test_pnl = pnl(y_test.values, test_labels, X_test["loan_amount"].values, X_test["interest_rate"].values)

print(f"ROC AUC : {roc_auc:.4f}")
print(f"Gini    : {gini:.4f}")
print(f"PR AUC  : {pr_auc:.4f}")
print(f"KS      : {ks:.4f}")
print(f"P&L on test_eval_20k : {test_pnl:,.0f}")

ROC AUC : 0.9157
Gini    : 0.8315
PR AUC  : 0.9729
KS      : 0.6656
P&L on test_eval_20k : 17,275,194


In [10]:
def fairness_report(X, y_true, y_pred, attr):
    rows = []
    for g, idx in X.groupby(attr, observed=True).groups.items():
        yt, yp = y_true.loc[idx], y_pred.loc[idx]
        pos, neg = (yt == 1), (yt == 0)
        rows.append({
            "group": g,
            "n": len(idx),
            "approval_rate": yp.mean(),
            "TPR": yp[pos].mean() if pos.sum() > 0 else np.nan,
            "FPR": yp[neg].mean() if neg.sum() > 0 else np.nan,
        })
    rep = pd.DataFrame(rows).set_index("group")
    rep["disparate_impact_ratio"] = rep["approval_rate"] / rep["approval_rate"].max()
    return rep.round(4)

y_test_s = pd.Series(y_test.values, index=X_test.index)
pred_s = pd.Series(test_labels, index=X_test.index)

for attr in PROTECTED_ATTRS:
    print(f"\n=== {attr} ===")
    print(fairness_report(X_test, y_test_s, pred_s, attr))


=== gender ===
            n  approval_rate     TPR     FPR  disparate_impact_ratio
group                                                               
Female  10155         0.7664  0.8959  0.2436                  0.9135
Male     9727         0.7545  0.8929  0.2166                  0.8993
Other     118         0.8390  0.9300  0.3333                  1.0000

=== marital_status ===
             n  approval_rate     TPR     FPR  disparate_impact_ratio
group                                                                
Divorced   762         0.7480  0.8592  0.2980                  0.9780
Married   9294         0.7589  0.8932  0.2320                  0.9921
Single    9757         0.7649  0.8988  0.2241                  1.0000
Widowed    187         0.7219  0.8921  0.2292                  0.9438

=== employment_status ===
                   n  approval_rate     TPR     FPR  disparate_impact_ratio
group                                                                      
Employed       1

In [11]:
from scipy.stats import norm

def tost_proportions(approved_a, n_a, approved_b, n_b, delta, alpha=0.05):
    """
    Two-One-Sided T-Test (Schuirmann 1987)，课程8.3节提到的方法。
    H0: |p_a - p_b| >= delta  (不公平)
    H1: |p_a - p_b| <  delta  (公平：差距在容忍度delta以内)

    拒绝H0 (p_tost < alpha) 才叫"证明了公平"，
    没拒绝H0 只代表"没能证明公平"，不代表模型不公平——这就是课程强调的方向反转。
    """
    p_a, p_b = approved_a / n_a, approved_b / n_b
    diff = p_a - p_b
    se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)

    z_low = (diff - (-delta)) / se   # 检验 diff <= -delta
    z_high = (diff - delta) / se     # 检验 diff >= delta
    p_low = 1 - norm.cdf(z_low)
    p_high = norm.cdf(z_high)
    p_tost = max(p_low, p_high)      # TOST取两个单侧检验里较大的p值

    return {
        "diff": diff, "p_tost": p_tost,
        "certified_equivalent": p_tost < alpha,   # True = 证明了公平
    }

DELTA = 0.02  # 容忍度2个百分点，和你组员用的一致，方便横向对比

# 对gender做两两比较（Male vs Female，样本量最大的两组）
def approval_stats(df, col, group):
    sub = df[df[col] == group]
    return sub["approved"].sum(), len(sub)

results_gender = X_test.assign(approved=test_labels)
a, n_a = approval_stats(results_gender, "gender", "Male")
b, n_b = approval_stats(results_gender, "gender", "Female")
print("Gender (Male vs Female):", tost_proportions(a, n_a, b, n_b, delta=DELTA))

# marital_status 用两两组合循环跑
from itertools import combinations
groups = X_test["marital_status"].cat.categories if hasattr(X_test["marital_status"], "cat") else X_test["marital_status"].unique()
for g1, g2 in combinations(groups, 2):
    a, n_a = approval_stats(results_gender, "marital_status", g1)
    b, n_b = approval_stats(results_gender, "marital_status", g2)
    res = tost_proportions(a, n_a, b, n_b, delta=DELTA)
    print(f"marital_status ({g1} vs {g2}):", res)

Gender (Male vs Female): {'diff': np.float64(-0.011922692863271744), 'p_tost': np.float64(0.0911295699161001), 'certified_equivalent': np.False_}
marital_status (Divorced vs Married): {'diff': np.float64(-0.010845198578712245), 'p_tost': np.float64(0.2876629759256506), 'certified_equivalent': np.False_}
marital_status (Divorced vs Single): {'diff': np.float64(-0.016855251912820113), 'p_tost': np.float64(0.42352008590698875), 'certified_equivalent': np.False_}
marital_status (Divorced vs Widowed): {'diff': np.float64(0.02610636237315256), 'p_tost': np.float64(0.5667148197821125), 'certified_equivalent': np.False_}
marital_status (Married vs Single): {'diff': np.float64(-0.006010053334107868), 'p_tost': np.float64(0.011728625541086779), 'certified_equivalent': np.True_}
marital_status (Married vs Widowed): {'diff': np.float64(0.036951560951864804), 'p_tost': np.float64(0.6959168717901587), 'certified_equivalent': np.False_}
marital_status (Single vs Widowed): {'diff': np.float64(0.042961

In [12]:
from scipy.stats import chi2_contingency

def cramers_v(a, b):
    """类别 vs 类别 的关联强度，0=无关联，1=完全关联"""
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

def eta_squared(cat, num):
    """数值 vs 类别 的关联强度（组间方差占比），0=无关联，1=完全由该类别决定"""
    groups = [num[cat == g] for g in cat.unique()]
    grand_mean = num.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((num - grand_mean) ** 2).sum()
    return ss_between / ss_total

CAT_CANDIDATES = ["employment_status", "education_level", "loan_purpose", "grade_subgrade"]
NUM_CANDIDATES = ["annual_income", "debt_to_income_ratio", "credit_score", "loan_amount", "interest_rate"]
PROTECTED = ["gender", "marital_status"]

for p in PROTECTED:
    print(f"\n=== {p} 的潜在代理变量检查 ===")
    for c in CAT_CANDIDATES:
        print(f"  Cramér's V vs {c:22s}: {cramers_v(X_test[p], X_test[c]):.4f}")
    for n in NUM_CANDIDATES:
        print(f"  Eta²        vs {n:22s}: {eta_squared(X_test[p], X_test[n]):.4f}")


=== gender 的潜在代理变量检查 ===
  Cramér's V vs employment_status     : 0.0141
  Cramér's V vs education_level       : 0.0102
  Cramér's V vs loan_purpose          : 0.0245
  Cramér's V vs grade_subgrade        : 0.0384
  Eta²        vs annual_income         : 0.0000
  Eta²        vs debt_to_income_ratio  : 0.0001
  Eta²        vs credit_score          : 0.0001
  Eta²        vs loan_amount           : 0.0001
  Eta²        vs interest_rate         : 0.0000

=== marital_status 的潜在代理变量检查 ===
  Cramér's V vs employment_status     : 0.0097
  Cramér's V vs education_level       : 0.0118
  Cramér's V vs loan_purpose          : 0.0232
  Cramér's V vs grade_subgrade        : 0.0383
  Eta²        vs annual_income         : 0.0001
  Eta²        vs debt_to_income_ratio  : 0.0001
  Eta²        vs credit_score          : 0.0005
  Eta²        vs loan_amount           : 0.0000
  Eta²        vs interest_rate         : 0.0002


In [13]:
deltas = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10]

print("Gender (Male vs Female):")
for d in deltas:
    res = tost_proportions(a, n_a, b, n_b, delta=d)
    print(f"  delta={d:.3f} -> diff={res['diff']:.4f}, p_tost={res['p_tost']:.4f}, "
          f"certified={res['certified_equivalent']}")

Gender (Male vs Female):
  delta=0.005 -> diff=0.0430, p_tost=0.8747, certified=False
  delta=0.010 -> diff=0.0430, p_tost=0.8407, certified=False
  delta=0.020 -> diff=0.0430, p_tost=0.7564, certified=False
  delta=0.030 -> diff=0.0430, p_tost=0.6526, certified=False
  delta=0.050 -> diff=0.0430, p_tost=0.4157, certified=False
  delta=0.100 -> diff=0.0430, p_tost=0.0422, certified=True
